<a href="https://colab.research.google.com/github/VivekGaddam/EDAVassignment/blob/main/EDAVAssignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/StudentsPerformance.csv')
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


## Q1. Score Distributions & Variance Analysis
Load the dataset, create a **Series** for math, reading, and writing scores.  
Use `describe()` to examine distributions and identify the subject with the **highest variance**.

In [ ]:
math_scores    = df['math score']
reading_scores = df['reading score']
wri ting_scores = df['writing score']

print('=' * 55)
print('           MATH SCORE — describe()')
print('=' * 55)
print(math_scores.describe())
print('\n' + '=' * 55)
print('        READING SCORE — describe()')
print('=' * 55)
print(reading_scores.describe())

print('\n' + '=' * 55)
print('        WRITING SCORE — describe()')
print('=' * 55)
print(writing_scores.describe())

           MATH SCORE — describe()
count    1000.00000
mean       66.08900
std        15.16308
min         0.00000
25%        57.00000
50%        66.00000
75%        77.00000
max       100.00000
Name: math score, dtype: float64

        READING SCORE — describe()
count    1000.000000
mean       69.169000
std        14.600192
min        17.000000
25%        59.000000
50%        70.000000
75%        79.000000
max       100.000000
Name: reading score, dtype: float64

        WRITING SCORE — describe()
count    1000.000000
mean       68.054000
std        15.195657
min        10.000000
25%        57.750000
50%        69.000000
75%        79.000000
max       100.000000
Name: writing score, dtype: float64


In [ ]:
# Compute variances
variances = pd.Series({
    'math score':    math_scores.var(),
    'reading score': reading_scores.var(),
    'writing score': writing_scores.var()
})

print('\nVariance by Subject:')
print(variances)

highest_var_subject = variances.idxmax()
print(f'\n Subject with HIGHEST variance: {highest_var_subject} ({variances.max():.2f})')



Variance by Subject:
math score       229.918998
reading score    213.165605
writing score    230.907992
dtype: float64

 Subject with HIGHEST variance: writing score (230.91)


## Q2. Effect of Test Preparation Course
Use `groupby()` on `test preparation course` with index-aligned operations to compare mean scores.  
Determine if preparation **significantly improves** performance across all three subjects.

In [ ]:
# Group by test preparation course
prep_group = df.groupby('test preparation course')[['math score', 'reading score', 'writing score']]

prep_mean = prep_group.mean()
prep_std  = prep_group.std()

print('📊 Mean Scores by Test Preparation Course:')
print(prep_mean.round(2))

print('\n📊 Std Deviation:')
print(prep_std.round(2))

# Index-aligned difference
diff = prep_mean.loc['completed'] - prep_mean.loc['none']
print('\n📈 Score Improvement (completed − none):')
print(diff.round(2))

# % improvement
pct_improvement = (diff / prep_mean.loc['none']) * 100
print('\n📈 % Improvement:')
print(pct_improvement.round(2))

# Verdict
all_positive = all(diff > 0)
print(f'\n🎯 Does preparation significantly improve ALL subjects? → {"YES ✅" if all_positive else "NO ❌"}')



📊 Mean Scores by Test Preparation Course:
                         math score  reading score  writing score
test preparation course                                          
completed                     69.70          73.89          74.42
none                          64.08          66.53          64.50

📊 Std Deviation:
                         math score  reading score  writing score
test preparation course                                          
completed                     14.44          13.64          13.38
none                          15.19          14.46          15.00

📈 Score Improvement (completed − none):
math score       5.62
reading score    7.36
writing score    9.91
dtype: float64

📈 % Improvement:
math score        8.77
reading score    11.06
writing score    15.37
dtype: float64

🎯 Does preparation significantly improve ALL subjects? → YES ✅


## Q3. Hierarchical Indexing — Education & Gender
Apply **hierarchical indexing** using `parental level of education` and `gender`.  
Examine average scores using `xs()` and `loc[]`. Identify patterns between education level and performance.

In [ ]:
# Build hierarchical index
df_hier = df.set_index(['parental level of education', 'gender'])
score_cols = ['math score', 'reading score', 'writing score']

# Average scores at each level
avg_scores = df_hier[score_cols].groupby(level=[0, 1]).mean().round(2)

print('📊 Average Scores by Parental Education Level & Gender:')
print(avg_scores)

# Using xs() to slice a specific education level
edu_level = "master's degree"
print(f"\n🔍 xs() — Scores for '{edu_level}':")
print(avg_scores.xs(edu_level, level='parental level of education'))

# Using loc[]
print(f"\n🔍 loc[] — Scores for 'high school' > 'female':")
try:
    print(avg_scores.loc[('high school', 'female')])
except:
    print(avg_scores.xs('high school', level=0).loc['female'])

# Education-performance pattern
edu_avg = df.groupby('parental level of education')[score_cols].mean().mean(axis=1).sort_values(ascending=False)
print('\n📈 Average Overall Score by Education Level (sorted):')
print(edu_avg.round(2))

edu_order = ["some high school", "high school", "some college",
             "associate's degree", "bachelor's degree", "master's degree"]
df_edu = df.groupby('parental level of education')[score_cols].mean().reindex(
    [e for e in edu_order if e in df['parental level of education'].unique()])




📊 Average Scores by Parental Education Level & Gender:
                                    math score  reading score  writing score
parental level of education gender                                          
associate's degree          female       65.25          74.12          74.00
                            male         70.76          67.43          65.41
bachelor's degree           female       68.35          77.29          78.38
                            male         70.58          68.09          67.65
high school                 female       59.35          68.20          66.69
                            male         64.71          61.48          58.54
master's degree             female       66.50          76.81          77.64
                            male         74.83          73.13          72.61
some college                female       65.41          73.55          74.05
                            male         69.01          64.99          63.15
some high school     

## Q4. Academic Performance Index (API)
Create a **weighted composite API** (Math 40%, Reading 30%, Writing 30%) using ufunc operations.  
Rank all students and display a detailed attribute breakdown of the **top and bottom performers**.

In [ ]:
# Weighted API: Math 40%, Reading 30%, Writing 30%
weights = np.array([0.40, 0.30, 0.30])
scores_matrix = df[['math score', 'reading score', 'writing score']].values

# ufunc-style weighted operation using np.dot
df['API'] = np.dot(scores_matrix, weights)

# Rank students (1 = best)
df['Rank'] = df['API'].rank(ascending=False, method='min').astype(int)

print('📊 API Statistics:')
print(df['API'].describe().round(2))

# Top 10 performers
top10 = df.nlargest(10, 'API')[['gender','race/ethnicity','parental level of education',
                                 'test preparation course','math score','reading score',
                                 'writing score','API','Rank']]
print('\n🏆 TOP 10 Performers:')
print(top10.to_string(index=False))

# Bottom 10 performers
bot10 = df.nsmallest(10, 'API')[['gender','race/ethnicity','parental level of education',
                                   'test preparation course','math score','reading score',
                                   'writing score','API','Rank']]
print('\n⚠️  BOTTOM 10 Performers:')
print(bot10.to_string(index=False))


📊 API Statistics:
count    1000.00
mean       67.60
std        14.24
min         8.10
25%        58.10
50%        68.30
75%        77.60
max       100.00
Name: API, dtype: float64

🏆 TOP 10 Performers:
gender race/ethnicity parental level of education test preparation course  math score  reading score  writing score   API  Rank
female        group E           bachelor's degree                    none         100            100            100 100.0     1
  male        group E           bachelor's degree               completed         100            100            100 100.0     1
female        group E          associate's degree                    none         100            100            100 100.0     1
female        group E           bachelor's degree               completed          99            100            100  99.6     4
female        group D                some college                    none          98            100             99  98.9     5
female        group D         

## Q5. Null Value Imputation Strategies
Introduce controlled nulls in **10% of rows**.  
Compare **mean, median, and mode** imputation strategies and assess their impact on group-level statistics.

In [ ]:
np.random.seed(123)

# Work on a copy
df_null = df[['gender','test preparation course','math score','reading score','writing score']].copy()

# Introduce 10% nulls
n_rows = len(df_null)
null_idx = np.random.choice(n_rows, size=int(0.1 * n_rows), replace=False)

for col in ['math score', 'reading score', 'writing score']:
    chosen = np.random.choice(null_idx, size=len(null_idx)//2, replace=False)
    df_null.loc[chosen, col] = np.nan

print(f'🔴 Null counts after introduction:')
print(df_null.isnull().sum())
print(f'\nTotal nulls: {df_null.isnull().sum().sum()}')

score_cols_q5 = ['math score', 'reading score', 'writing score']

# Strategy 1: Mean imputation
df_mean = df_null.copy()
for col in score_cols_q5:
    df_mean[col].fillna(df_mean[col].mean(), inplace=True)

# Strategy 2: Median imputation
df_median = df_null.copy()
for col in score_cols_q5:
    df_median[col].fillna(df_median[col].median(), inplace=True)

# Strategy 3: Mode imputation
df_mode = df_null.copy()
for col in score_cols_q5:
    df_mode[col].fillna(df_mode[col].mode()[0], inplace=True)

# Compare group-level statistics (by test preparation course)
def group_stats(frame, name):
    return frame.groupby('test preparation course')[score_cols_q5].mean().round(2).assign(Strategy=name)

stats_comparison = pd.concat([
    group_stats(df_null,   'Original (with nulls)'),
    group_stats(df_mean,   'Mean Imputed'),
    group_stats(df_median, 'Median Imputed'),
    group_stats(df_mode,   'Mode Imputed')
])

print('\n📊 Group-Level Mean Scores Under Each Imputation Strategy:')
print(stats_comparison.to_string())


🔴 Null counts after introduction:
gender                      0
test preparation course     0
math score                 50
reading score              50
writing score              50
dtype: int64

Total nulls: 150

📊 Group-Level Mean Scores Under Each Imputation Strategy:
                         math score  reading score  writing score               Strategy
test preparation course                                                                 
completed                     69.52          73.72          74.41  Original (with nulls)
none                          64.08          66.43          64.46  Original (with nulls)
completed                     69.38          73.48          74.18           Mean Imputed
none                          64.19          66.56          64.67           Mean Imputed
completed                     69.38          73.53          74.21         Median Imputed
none                          64.18          66.61          64.72         Median Imputed
completed     

## Q6. Boolean Indexing — High Performers
Use **boolean indexing** to filter students scoring above 90 in all three subjects.  
Examine their demographic profile using `value_counts()` and cross-tabulation.

In [ ]:
# Boolean mask — all three scores > 90
mask = (df['math score'] > 90) & (df['reading score'] > 90) & (df['writing score'] > 90)
top_students = df[mask].copy()

print(f'📊 Students scoring >90 in ALL subjects: {len(top_students)} ({len(top_students)/len(df)*100:.1f}%)')
print(top_students[['math score','reading score','writing score','gender','race/ethnicity',
                      'parental level of education','test preparation course']].to_string())

print('\n📊 Gender Distribution (value_counts):')
print(top_students['gender'].value_counts())

print('\n📊 Race/Ethnicity Distribution:')
print(top_students['race/ethnicity'].value_counts())

print('\n📊 Test Preparation Course:')
print(top_students['test preparation course'].value_counts())

print('\n📊 Cross-tabulation — Gender × Test Prep:')
print(pd.crosstab(top_students['gender'], top_students['test preparation course']))

print('\n📊 Cross-tabulation — Race × Education Level:')
print(pd.crosstab(top_students['race/ethnicity'], top_students['parental level of education']))


📊 Students scoring >90 in ALL subjects: 23 (2.3%)
     math score  reading score  writing score  gender race/ethnicity parental level of education test preparation course
114          99            100            100  female        group E           bachelor's degree               completed
149         100            100             93    male        group E          associate's degree               completed
165          96            100            100  female        group C           bachelor's degree               completed
179          97            100            100  female        group D            some high school               completed
451         100             92             97  female        group E                some college                    none
458         100            100            100  female        group E           bachelor's degree                    none
546          92            100             97  female        group A            some high school       

## Q7. Grading Function & Grade Report
Design a grading function using `apply()` to assign **letter grades (A–F)** per subject.  
Build a full grade report DataFrame with **hierarchical indexing** on gender and race/ethnicity.

In [ ]:
# Grading function
def assign_grade(score):
    if   score >= 90: return 'A'
    elif score >= 80: return 'B'
    elif score >= 70: return 'C'
    elif score >= 60: return 'D'
    else:             return 'F'

# Apply to each subject using apply()
df['math grade']    = df['math score'].apply(assign_grade)
df['reading grade'] = df['reading score'].apply(assign_grade)
df['writing grade'] = df['writing score'].apply(assign_grade)

# Grade distribution per subject
print('📊 Grade Distribution — Math:')
print(df['math grade'].value_counts().sort_index())
print('\n📊 Grade Distribution — Reading:')
print(df['reading grade'].value_counts().sort_index())
print('\n📊 Grade Distribution — Writing:')
print(df['writing grade'].value_counts().sort_index())

# Build grade report with hierarchical index on gender and race/ethnicity
grade_report = df[['gender', 'race/ethnicity',
                    'math grade', 'reading grade', 'writing grade']].copy()
grade_report = grade_report.set_index(['gender', 'race/ethnicity'])

print('\n📋 Grade Report (hierarchical index — sample 15 rows):')
print(grade_report.head(15))

# Grade distribution heatmap per group




📊 Grade Distribution — Math:
math grade
A     58
B    135
C    216
D    268
F    323
Name: count, dtype: int64

📊 Grade Distribution — Reading:
reading grade
A     79
B    170
C    264
D    233
F    254
Name: count, dtype: int64

📊 Grade Distribution — Writing:
writing grade
A     78
B    157
C    254
D    230
F    281
Name: count, dtype: int64

📋 Grade Report (hierarchical index — sample 15 rows):
                      math grade reading grade writing grade
gender race/ethnicity                                       
female group B                 C             C             C
       group C                 D             A             B
       group B                 A             A             A
male   group A                 F             F             F
       group C                 C             C             C
female group B                 C             B             C
       group B                 B             A             A
male   group B                 F             F   

## Q8. Index Alignment & Underperformance
Compute the difference between each student's scores and the **class average** using index-aligned subtraction.  
Identify students who **underperform** in specific subjects.

In [ ]:
score_cols = ['math score', 'reading score', 'writing score']

# Class average (Series)
class_avg = df[score_cols].mean()
print('📊 Class Averages:')
print(class_avg.round(2))

# Deviation per student — index alignment via subtraction
df_scores = df[score_cols].copy()
deviation  = df_scores.sub(class_avg, axis='columns')  # aligned by column name

print('\n📊 Score Deviation (sample 10 rows):')
print(deviation.head(10).round(2))

print('\n📊 Deviation Statistics:')
print(deviation.describe().round(2))

# Identify underperformers: below average in at least 2 subjects
below_avg = (deviation < 0).sum(axis=1)
underperformers = df[below_avg >= 2][['gender', 'race/ethnicity',
                                        'parental level of education',
                                        'math score', 'reading score', 'writing score']]

print(f'\n⚠️  Students below average in ≥2 subjects: {len(underperformers)} ({len(underperformers)/len(df)*100:.1f}%)')

# Students below average in all 3
all_below = df[(deviation < 0).all(axis=1)]
print(f'❗ Students below average in ALL 3 subjects: {len(all_below)} ({len(all_below)/len(df)*100:.1f}%)')


📊 Class Averages:
math score       66.09
reading score    69.17
writing score    68.05
dtype: float64

📊 Score Deviation (sample 10 rows):
   math score  reading score  writing score
0        5.91           2.83           5.95
1        2.91          20.83          19.95
2       23.91          25.83          24.95
3      -19.09         -12.17         -24.05
4        9.91           8.83           6.95
5        4.91          13.83           9.95
6       21.91          25.83          23.95
7      -26.09         -26.17         -29.05
8       -2.09          -5.17          -1.05
9      -28.09          -9.17         -18.05

📊 Deviation Statistics:
       math score  reading score  writing score
count     1000.00        1000.00        1000.00
mean         0.00           0.00          -0.00
std         15.16          14.60          15.20
min        -66.09         -52.17         -58.05
25%         -9.09         -10.17         -10.30
50%         -0.09           0.83           0.95
75%         10.9

## Q9. DataFrame Operations — Prep vs No-Prep
Split students into two DataFrames by preparation status.  
Use **subtraction and percentage operations** with aligned indices to quantify score differences between groups.

In [ ]:
score_cols = ['math score', 'reading score', 'writing score']

# Split into two DataFrames
df_completed = df[df['test preparation course'] == 'completed'][score_cols].reset_index(drop=True)
df_none      = df[df['test preparation course'] == 'none'][score_cols].reset_index(drop=True)

print(f'📊 Completed preparation: {len(df_completed)} students')
print(f'📊 No preparation:        {len(df_none)} students')

# Group-level mean for aligned operations
mean_completed = df_completed.mean()
mean_none      = df_none.mean()

# Subtraction with index alignment
score_diff = mean_completed.sub(mean_none)
print('\n📈 Mean Score Difference (completed − none):')
print(score_diff.round(2))

# Percentage operation
pct_change = (score_diff / mean_none) * 100
print('\n📈 % Change:')
print(pct_change.round(2))

# Std deviation comparison
std_diff = df_completed.std().sub(df_none.std())
print('\n📊 Std Deviation Difference (completed − none):')
print(std_diff.round(2))

# Variance ratio
var_ratio = df_completed.var() / df_none.var()
print('\n📊 Variance Ratio (completed / none):')
print(var_ratio.round(3))



📊 Completed preparation: 358 students
📊 No preparation:        642 students

📈 Mean Score Difference (completed − none):
math score       5.62
reading score    7.36
writing score    9.91
dtype: float64

📈 % Change:
math score        8.77
reading score    11.06
writing score    15.37
dtype: float64

📊 Std Deviation Difference (completed − none):
math score      -0.75
reading score   -0.83
writing score   -1.62
dtype: float64

📊 Variance Ratio (completed / none):
math score       0.904
reading score    0.889
writing score    0.795
dtype: float64


## Q10. Comprehensive Analytics Report
Build a full analytics report covering **null analysis, hierarchical grouping, ufunc transformations, and grade distribution**.  
Visualize all key insights in a single multi-panel figure.

In [ ]:
print('=' * 60)
print('   COMPREHENSIVE STUDENT ANALYTICS REPORT')
print('=' * 60)

print('\n── 1. NULL ANALYSIS ──')
print(f'Dataset shape: {df.shape}')
print(f'Total null values: {df.isnull().sum().sum()}')
print(df.isnull().sum())

print('\n── 2. DATASET OVERVIEW ──')
print(df[['math score','reading score','writing score','API']].describe().round(2))

   COMPREHENSIVE STUDENT ANALYTICS REPORT

── 1. NULL ANALYSIS ──
Dataset shape: (1000, 13)
Total null values: 0
gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
API                            0
Rank                           0
math grade                     0
reading grade                  0
writing grade                  0
dtype: int64

── 2. DATASET OVERVIEW ──
       math score  reading score  writing score      API
count     1000.00        1000.00        1000.00  1000.00
mean        66.09          69.17          68.05    67.60
std         15.16          14.60          15.20    14.24
min          0.00          17.00          10.00     8.10
25%         57.00          59.00          57.75    58.10
50%         66.00          70.00          69.00    68.30
75%         77.00  